# vidaudit — FineVideo evaluation

This notebook is the project's **reproduction artifact** (PLAN.md step 11/12, DESIGN.md DD-13/DD-15). It:

1. Streams a handful of videos from [FineVideo](https://huggingface.co/datasets/HuggingFaceFV/finevideo) and extracts their ground-truth chapter descriptions.
2. Builds a labeled dataset of **plausible synthetic mutations** (object swap, attribute change, entity injection) plus a **real-hallucination subset** harvested by captioning frames.
3. Runs **vidaudit** (canonical Qwen2.5-VL-3B backend) and the **text-comparison baseline** over the dataset.
4. Sweeps the confidence threshold, then prints precision / recall / F1 **split by subset**, plus extraction recall.

**Runtime:** T4 GPU (Runtime → Change runtime type → T4 GPU).

**Reproducibility:** FineVideo is gated — you'll need to accept its terms and log in with `huggingface_hub`. Pin the Qwen revision (`--qwen-revision` / `revision=`) when you report numbers (DD-14).

## 1. Install + authenticate

In [ ]:
!git clone https://github.com/shaneopatrick/vidaudit.git /content/vidaudit
%cd /content/vidaudit
!pip install -q -e '.[qwen,eval]'
!python -m spacy download en_core_web_sm

In [ ]:
# FineVideo is a gated dataset — accept terms at
# https://huggingface.co/datasets/HuggingFaceFV/finevideo then log in.
from huggingface_hub import notebook_login

notebook_login()

## 2. Stream FineVideo → chapters + on-disk videos

We iterate the stream manually so each video's `.mp4` is written to disk with a `video_id` that matches the chapters extracted from its metadata — the frame sampler needs the file on disk.

In [ ]:
from pathlib import Path
from datasets import load_dataset
from eval.finevideo_loader import FineVideoChapter

LIMIT = 5
VIDEOS = Path('/content/videos'); VIDEOS.mkdir(exist_ok=True)

def _tc(v):
    if isinstance(v, (int, float)): return float(v)
    if not isinstance(v, str) or not v.strip(): return None
    s = v.strip()
    try: return float(s)
    except ValueError: pass
    sec = 0.0
    for p in s.split(':'):
        try: sec = sec * 60 + float(p)
        except ValueError: return None
    return sec

def _scene_desc(scene):
    if scene.get('description'): return scene['description']
    acts = scene.get('activities') or []
    joined = ' '.join(a.get('description', '') for a in acts if isinstance(a, dict)).strip()
    return joined or scene.get('title', '')

def chapters_from_row(row, video_id):
    meta = row.get('json') or {}
    content = meta.get('content_metadata', meta)
    out = []
    for sc in content.get('scenes', []):
        ts = sc.get('timestamps') if isinstance(sc.get('timestamps'), dict) else {}
        start = _tc(ts.get('start_timestamp'))
        desc = _scene_desc(sc)
        if start is None or not desc: continue
        out.append(FineVideoChapter(
            video_id=video_id, timestamp_start=start,
            timestamp_end=_tc(ts.get('end_timestamp')), description=desc))
    return out

def _video_bytes(row):
    v = row.get('mp4') or row.get('video')
    if isinstance(v, dict): return v.get('bytes')
    if isinstance(v, dict): return v.get('bytes')
    return v if isinstance(v, (bytes, bytearray)) else None

ds = load_dataset('HuggingFaceFV/finevideo', split='train', streaming=True)
chapters = []
for i, row in enumerate(ds):
    if i >= LIMIT: break
    vid = f'video_{i}'
    data = _video_bytes(row)
    if data: (VIDEOS / f'{vid}.mp4').write_bytes(data)
    chapters.extend(chapters_from_row(row, vid))
print(f'{len(chapters)} chapters')
if chapters: 
    print(chapters[0].model_dump())

If you got **0 chapters**, FineVideo's metadata layout may differ from what `chapters_from_row` expects. Inspect one row's `json` and adjust the key paths in `eval/finevideo_loader.py`:

```python
import itertools, json
row = next(itertools.islice(load_dataset('HuggingFaceFV/finevideo', split='train', streaming=True), 1))
print(json.dumps(row['json'], indent=2)[:2000])
```

## 3. Build the labeled dataset

Synthetic mutations are deterministic and auto-labeled. The real subset is harvested by captioning each chapter's frame — those samples need **hand-labeling** (`real_is_hallucinated`) before they contribute to metrics.

In [ ]:
from eval.finevideo_loader import make_synthetic_samples, save_dataset

samples = []
for chapter in chapters:
    samples.extend(make_synthetic_samples(chapter))

print(f'{len(samples)} synthetic samples')
save_dataset(samples, Path('/content/eval_dataset.json'))

In [ ]:
# Load the canonical backend once; reuse its runner as the captioner (no second load).
from eval.captioner import qwen_captioner
from eval.run_eval import make_frame_for
from vidaudit.vlm.qwen_vl import QwenVLBackend

backend = QwenVLBackend()  # add revision='<sha>' to pin (DD-14)
frame_for = make_frame_for(VIDEOS)
captioner = qwen_captioner(backend._runner)

In [ ]:
# Harvest the real subset (captioner's natural errors). These come back
# UNLABELED — set `real_is_hallucinated` by hand to score them (DD-13).
from eval.finevideo_loader import build_real_samples


def primary_frame(chapter):
    # A FineVideoChapter exposes video_id/timestamp_start/timestamp_end, so it
    # duck-types as the sample make_frame_for expects. Take the primary frame.
    frames = frame_for(chapter)
    return frames[0] if frames else None


real_samples = build_real_samples(chapters, captioner, frame_for=primary_frame)
print(f'{len(real_samples)} real samples harvested (label them before scoring).')
for s in real_samples[:3]:
    print('GT  :', s.clean_description)
    print('CAP :', s.mutated_description)
    print()

## 4. Run the eval

The confidence sweep re-scores at each threshold; thanks to the verification cache (DD-11) only the first pass spends compute. The synthetic subset is scored automatically; add your hand-labeled real samples to `all_samples` once labeled.

In [ ]:
from eval.run_eval import make_baseline_caption_for, make_vidaudit_auditor, render_eval_report, run_eval

auditor = make_vidaudit_auditor(backend, frame_for)
baseline_caption_for = make_baseline_caption_for(captioner, frame_for)

all_samples = samples  # + [labeled real samples]
report = run_eval(all_samples, auditor, baseline_caption_for)

render_eval_report(report)
report.save_json(Path('/content/eval_report.json'))
print('\nSaved /content/eval_report.json')

## 5. Threshold sweep

The shipped default confidence threshold (DD-12) should be the best-F1 point here, not an asserted constant.

In [ ]:
print(f'{"conf":>6} {"P":>6} {"R":>6} {"F1":>6}')
for p in report.sweep:
    c = p.confusion
    print(f'{p.confidence_threshold:6.2f} {c.precision:6.2f} {c.recall:6.2f} {c.f1:6.2f}')
print(f'\nbest threshold: {report.best_confidence_threshold:.2f}')
print(f'extraction recall: {report.extraction_recall:.2f}')